# ECE326 Tutorial 1: Lab Introduction


## Tutorial 1: From Source to Service

Wednesday, September 23, 2026, 9:00-10:00 AM.

Today we will run the local Lab 1 service, send it a request, and inspect a
benchmark. This is an unmarked practice notebook, not a completed lab report.
Use the Lab 1 handout for assessed requirements and Quercus for course policies.
Lab 1 is due September 27 at 11:59 PM.

Labs 1 and 2 are entirely local. CloudMake is introduced in Lab 3.

| Minutes | Activity |
|---------|----------|
| 0-8     | Check the starter and run the tests |
| 8-15    | Generate text; practise shapes and loop classification |
| 15-30   | Send and inspect one live HTTP request |
| 30-42   | Run the benchmark and inspect its measurements |
| 42-50   | State-ownership exercise and exit check |
| 50-60   | Questions and setup troubleshooting |


### Before the tutorial

Download the Lab 1 starter ZIP and the Project/Lab 1 instructions from the course
release. Extract the ZIP once into your working directory. Its project root is
`ece326-paradigmmorph/`, containing `Makefile`, `tiny_llama/`, and `tiny_serve/`.
Do not use a TA solution directory.

Complete the setup cells below before class.
Setup downloads packages and model weights and builds the reference C oracle;
it needs Internet access, Python 3.10 or newer, make, Git, and a C compiler.
The service uses Unix `fork`: use macOS or Linux. On Windows, arrange a local
Linux environment such as WSL with staff before the session; native Windows
does not support this worker model. No GPU or cloud account is needed.

Read the individual doctor results: missing downloads before setup can be
expected. After setup, investigate failures and unexpected skipped tests.
Keep the error text for staff rather than repeatedly reinstalling everything.

Open this notebook in JupyterLab with a Python 3 kernel. We use IPython's
`%cd` to change directories, `!make` to run project commands, and `%%writefile`
to create a request file. These cells are intended for Jupyter, not plain
Python or Org Babel execution. Run them in order. Setup installs dependencies,
the service cell starts a local process, and the cleanup cell stops it.


## 1. Find your project

Set `PROJECT_DIR` to the extracted project directory, not the ZIP and not its
parent. If this notebook is inside the project root, the default works.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import time
from urllib.request import Request, urlopen

PROJECT_DIR = Path(".").expanduser().resolve()  # Edit if the notebook is elsewhere.
BASE_URL = "http://127.0.0.1:8080"
PORT = 8080
PROJECT_PYTHON = sys.executable  # Or the absolute path to an existing lab Python.

print("Notebook Python:", sys.version.split()[0])
print("Notebook platform:", platform.platform())
print("Project directory:", PROJECT_DIR)
for name in ("Makefile", "tiny_llama", "tiny_serve", "models"):
    print(f"{name:12} {'found' if (PROJECT_DIR / name).exists() else 'not found: check path/setup'}")


Use `%cd` rather than `!cd`: the magic changes the working directory for later
cells, whereas a shell's directory change ends with that shell. Passing
`PROJECT_PYTHON` to make selects the project interpreter; it defaults to the
notebook's Python. For an existing lab environment, use its absolute Python path.


In [ ]:
assert (PROJECT_DIR / "Makefile").is_file(), "Set PROJECT_DIR to the extracted starter first."
%cd {PROJECT_DIR}
%pwd


In [ ]:
!make doctor PYTHON="{PROJECT_PYTHON}"


Run this setup cell before class. If setup is already complete, leave the flag
false. Set it to true when installing the starter for the first time.


In [ ]:
RUN_SETUP = False
if RUN_SETUP:
    !make setup PYTHON="{PROJECT_PYTHON}"
    assert _exit_code == 0, "Setup failed; inspect its output before continuing."
else:
    print("Using existing setup. For first-time installation, set RUN_SETUP = True.")


In [ ]:
!make test PYTHON="{PROJECT_PYTHON}"
assert _exit_code == 0, "Fix the test failure before continuing."


## 2. Generate text locally


In [ ]:
!make run MODEL=stories260K PROMPT="Once upon a time" TOKENS=16 PYTHON="{PROJECT_PYTHON}"
assert _exit_code == 0, "Inspect the generation error above."


The checkpoint `stories260K` is small and quick; its text quality is not the
goal. The larger `stories15M` checkpoint is used for the Lab 1 visible service
demonstration. Both are small Llama-style models trained on TinyStories, not
instruction-following assistants. A chat-shaped API does not make them chat models.


```text
prompt -> tokenizer -> token IDs -> NumPy model -> logits
       -> choose a token -> decode text -> extend the prefix -> repeat
```


Predict before discussing: does increasing `TOKENS` only change how much text
we print, or does it change how much computation we perform?

We will use the small checkpoint for the tutorial HTTP exercise as well.
For the assessed lab, follow the handout's requirement to demonstrate both
checkpoints and use `stories15M` for the canonical visible service demonstration.


### Shapes and loops: a small practice example

This toy batch contains two sequences, each with three token IDs. Predict its
shape and the shape of each row before running it. These are invented values,
not the checkpoint's shape ledger.


In [ ]:
toy_batch = [[11, 12, 13], [21, 22, 23]]
print("Batch shape:", (len(toy_batch), len(toy_batch[0])))
toy_shifted = [[token + 1 for token in row] for row in toy_batch]
print("Transformed batch:", toy_shifted)


Which axes do the two loops traverse? Does adding one to a token depend on the
previous token's result? Contrast this with generating the **next** token from
the growing prefix. Use that distinction when classifying the real model's
loops; you still need to inspect the project and derive its actual shapes.


## 3. One live request

The usual `!make serve` runs indefinitely and would hold this kernel busy.
Instead, `%%script bash --bg` runs the same service entry point in the background
so the following Python cells can send requests. The shell retains its child's
PID for cleanup; the shutdown cell below stops only this notebook's service.
Run the shutdown cell before rerunning the start cell.


In [ ]:
%env PYTHONPATH={PROJECT_DIR}:{PROJECT_DIR / '.deps'}
%env TUTORIAL_PYTHON={PROJECT_PYTHON}
%env TUTORIAL_PORT={PORT}
BASE_URL = f"http://127.0.0.1:{PORT}"


In [ ]:
%%script bash --bg --proc tutorial_service
trap 'exit 0' TERM INT
trap 'kill "$server_pid" 2>/dev/null; wait "$server_pid" 2>/dev/null' EXIT
"$TUTORIAL_PYTHON" -m tiny_serve --model stories260K --host 127.0.0.1 --port "$TUTORIAL_PORT" &
server_pid=$!
wait "$server_pid"


In [ ]:
for attempt in range(100):
    if tutorial_service.returncode is not None:
        raise RuntimeError("Service exited; check setup and whether the port is occupied.")
    try:
        with urlopen(BASE_URL + "/v1/models", timeout=0.5):
            break
    except OSError:
        time.sleep(0.1)
else:
    tutorial_service.terminate()
    raise TimeoutError("Service did not become ready. Check the startup output.")
print("Service ready:", BASE_URL)


Open [the local browser client](http://127.0.0.1:8080) for an interactive view.
The URL refers to the computer running the notebook kernel/client. Run both
locally for this tutorial. If port 8080 is occupied, change `PORT` in Section 1
to 8081 and rerun the configuration and service cells.


### Discover the API model name

The checkpoint selection (`stories260K`) and the advertised API name
(`paradigmmorph`) serve different purposes. Ask the service what it advertises.


In [ ]:
with urlopen(BASE_URL + "/v1/models", timeout=10) as response:
    model_listing = json.load(response)
print(json.dumps(model_listing, indent=2))


### Construct the request

The `%%writefile` magic writes the cell body to `tutorial-request.json` in the
project directory. Rerunning it replaces that practice file. JSON uses lowercase
`false`; Python uses `False`. Inspect the request before sending it.
What do `max_tokens`, `temperature`,
and `stream` control? Keep the prompt short enough for the small model's context.


In [ ]:
%%writefile tutorial-request.json
{
    "model": "paradigmmorph",
    "messages": [{"role": "user", "content": "Once upon a time"}],
    "max_tokens": 16,
    "temperature": 0,
    "stream": false
}


In [ ]:
payload = json.loads(Path("tutorial-request.json").read_text())
print(json.dumps(payload, indent=2))


### Send and observe


In [ ]:
request = Request(
    BASE_URL + "/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
started = time.perf_counter()
with urlopen(request, timeout=60) as response:
    reply = json.load(response)
    response_headers = dict(response.headers.items())
elapsed = time.perf_counter() - started
print(json.dumps(reply, indent=2))
print(f"Client elapsed: {elapsed:.3f} seconds")
print("Diagnostic headers:")
for name, value in response_headers.items():
    if name.lower().startswith("x-ece326-") or name.lower() == "server-timing":
        print(name + ":", value)


Record your own observations in the Markdown table below (double-click to edit).
Run the request twice: which fields remain the same, and which may differ?
A single elapsed time is an observation, not a performance conclusion.

| Observation | Your result |
|-------------|-------------|
| Advertised model | |
| Generated text | |
| Finish reason | |
| Prompt / completion token counts | |
| Worker PID on each run | |


### Follow the request


```text
client -> HTTP route -> validation and prompt construction
       -> model worker -> token generation -> complete JSON response -> client
```


Find the entry points in `tiny_serve/app.py` and `tiny_serve/protocol.py`.
For today, read the Bottle decorator as "register this function for this URL."
Worker processes and generator internals are supplied interfaces in Lab 1.

Discuss: the model generates tokens one at a time, but `stream=false` returns
one complete response. Is the client's first received byte evidence of when
the model computed its first token?

Stop the background service before continuing. Run this cleanup cell even if a
request fails. It is safe to run again after the service has exited.


In [ ]:
if tutorial_service.returncode is None:
    tutorial_service.terminate()
    for attempt in range(100):
        if tutorial_service.returncode is not None:
            break
        time.sleep(0.05)
    assert tutorial_service.returncode is not None, "Service cleanup has not completed."
print("Tutorial service stopped.")


## 4. Run and read the benchmark


In [ ]:
!make benchmark MODEL=stories260K PYTHON="{PROJECT_PYTHON}"
assert _exit_code == 0, "Inspect the benchmark error before reading its report."


This command starts its own local service, runs the fixture, saves a JSON report,
and stops that service. The fixture uses prerecorded conversations and simulated
client-side tool delays; it does not call an external AI service or real tools.

The report is `reports/fixture-stories260K.json`. Rerun the next cell after the
command finishes. If it is missing, check the command output and `PROJECT_DIR`.


In [ ]:
report_path = PROJECT_DIR / "reports" / "fixture-stories260K.json"
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Checkpoint:", report["checkpoint"])
    print("Fixture:", report["fixture_id"])
    for profile in report["profiles"]:
        summary = profile["summary"]
        print("\nAgent concurrency:", profile["active_concurrency"])
        for field in (
            "trajectories", "turns", "wall_seconds",
            "mean_inflight_requests", "peak_inflight_requests",
            "observed_tool_wait_seconds_total", "unique_worker_pids",
            "e2e_seconds", "server_model_ttft_seconds",
        ):
            print(f"  {field}: {summary.get(field, 'not recorded')}")
else:
    print("No report yet:", report_path)
    print("Run the !make benchmark cell above.")


### Three measurement layers

| Layer | Question |
|-------|----------|
| Model computation | How much numerical work does generating text require? |
| HTTP serving | How long does an active request take, and how many overlap? |
| Agent workload | How long does a whole sequence of requests and tool waits take? |

Discuss the observed report, rather than aiming for the TA's timing numbers:

- Are configured agent concurrency and mean in-flight requests equal? Why might they differ?
- What units do the fields above use? What do p50 and p95 summarize?
- Can a large tool-wait total establish that the model is slow?
- Can a nonstreaming response reveal client inter-token latency?


### Toy measurement exercise

These invented numbers describe one **sequential** trajectory, not the lab
fixture. Predict its elapsed time before running the cell. For concurrent
trajectories, summed waiting times need not equal wall-clock time.


In [ ]:
request_seconds = [0.10, 0.12, 0.08]
tool_wait_seconds = [0.50, 0.70]
request_total = sum(request_seconds)
wait_total = sum(tool_wait_seconds)
print("Time in requests:", round(request_total, 2), "s")
print("Time in tool waits:", round(wait_total, 2), "s")
print("Trajectory elapsed:", round(request_total + wait_total, 2), "s")


Which part would you investigate before claiming that a faster numerical kernel
will greatly improve the whole trajectory?


## 5. State ownership and exit check

Work with a neighbour, then write your own answers. This table is a practice
prompt, not text to paste into the assessed report.

| State | Who owns it? | When is it created? | When is it released or retained? |
|-------|--------------|---------------------|---------------------------------|
| Conversation history | | | |
| Parsed request body | | | |
| Loaded checkpoint arrays | | | |
| Current generation prefix | | | |
| Simulated tool-delay timer | | | |

Before leaving, show or explain:

1. A passing local test run, or the exact unresolved error and your next action.
2. One valid completion response.
3. Where conversation history lives between requests.
4. The measurement layer and units of one benchmark field.

The lab still requires your own program trace, shape ledger, loop classification,
both checkpoint demonstrations, and measurement interpretation. Use
`LAB1-REPORT.template.md` and `SUBMISSION.md` in your starter. Submit the required
benchmark evidence, but not model binaries, dependency directories, or caches.
This tutorial notebook is not a separate submission.


### Quick troubleshooting

- `No rule to make target`: check `PROJECT_DIR` and rerun the `%cd` cell.
- Missing dependencies or model files: finish `make setup` and inspect its error.
- Connection refused: run the background-service cell and check the URL and port.
- HTTP 400: check the model name, prompt length, and required payload fields.
- Skipped model/oracle tests: check downloaded assets and the C oracle setup.
- Weak stories260K output: expected; use correctness checks, not literary quality.
- Kernel restarted: rerun configuration and check whether a previous service still occupies the port.
